In [ ]:
# @title استخراج بيانات عميل Conquer Mobile فقط — فحص انتقائي دون إعادة الجرد
# خلية واحدة فقط. لا تعيد فحص Manifest ولا ملفات الصور/النماذج ولا بقية الأرشيف.
# تستخدم HTTP Range لقراءة فهرس ZIP وسحب ملفات APK الأربعة المحددة فقط.
import os, sys, json, csv, re, io, hashlib, zipfile, subprocess, textwrap, getpass, time
from pathlib import Path
from datetime import datetime, timezone

print("تهيئة فحص بيانات العميل المحدد فقط...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "remotezip"])
from remotezip import RemoteZip
import requests

SOURCE_URL = "https://github.com/ZzwahwahzZ/Conquer-Mobile-Assets./releases/download/mobile-extracted-assets/extracted-assets-001.zip"
SOURCE_BYTES = 842723106
SOURCE_SHA256 = "0d21a110c09b762ed5f1a5f4da022857b0534e489eb85052d15da460cb229d2a"
TARGET_REPO = "ZzwahwahzZ/Clash-of-Legends"
OUT = Path("/content/conquer_mobile_client_data_audit")
OUT.mkdir(parents=True, exist_ok=True)

# هذه هي ملفات العميل وحدها التي سبق التحقق من وجودها داخل الأرشيف.
TARGET_APKS = {
    "ConquerOnline/ConquerOnline-1.1.2.3.apk",
    "ConquerOnline/config.arm64_v8a.apk",
    "ConquerOnline/config.xxhdpi.apk",
    "ConquerOnline/install_time_1001.apk",
}
KEYWORDS = [
    "quest","mission","task","map","monster","npc","item","skill","equip",
    "class","profession","job","guild","clan","arena","dungeon","event",
    "shop","trade","market","refine","socket","gem","enchant","purify",
    "bless","soul","wing","pet","mount","drop","reward","teleport",
    "任务","地图","怪物","技能","装备","任务系统","副本","活动","商店","帮派"
]
TEXT_EXT = {".txt",".ini",".cfg",".json",".xml",".csv",".lua",".js",".yaml",".yml",".properties",".dat"}
MAX_TEXT_FILE = 2 * 1024 * 1024
MAX_TEXT_FILES_PER_APK = 160

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def classify(path):
    p = path.lower()
    groups = {
      "maps_world": ["map","scene","terrain","world","city","dungeon","teleport"],
      "quests_tasks": ["quest","mission","task","dialog","story","objective"],
      "characters_npc_monsters": ["npc","monster","mob","role","character","avatar"],
      "items_equipment": ["item","equip","weapon","armor","fashion","wing","mount","pet"],
      "skills_combat": ["skill","magic","spell","combat","effect","buff"],
      "economy_social": ["shop","trade","market","guild","clan","arena","event"],
      "game_config": ["config","setting","table","data","define","string","text"]
    }
    for category, words in groups.items():
        if any(w in p for w in words):
            return category
    return "other"

def candidate_hits(text):
    low = text.lower()
    return [k for k in KEYWORDS if k.lower() in low]

# احصل على رابط أصل الإصدار من GitHub API في كل تشغيل؛ لا نخزن رابط release-assets المؤقت.
# هذا يمنع استخدام رابط موقّع قديم انتهت صلاحيته. لا يتم تنزيل الأرشيف كاملًا.
API_HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "Clash-of-Legends-focused-client-audit",
    "Cache-Control": "no-cache"
}
session = requests.Session()
session.headers.update(API_HEADERS)
release_api = "https://api.github.com/repos/ZzwahwahzZ/Conquer-Mobile-Assets./releases/tags/mobile-extracted-assets"
release_response = session.get(release_api, timeout=45)
release_response.raise_for_status()
release = release_response.json()
asset = next((a for a in release.get("assets", []) if a.get("name") == "extracted-assets-001.zip"), None)
if not asset:
    raise RuntimeError("لم أجد extracted-assets-001.zip بالاسم المحدد داخل إصدار mobile-extracted-assets.")
if int(asset.get("size", -1)) != SOURCE_BYTES:
    raise RuntimeError(f"حجم أصل GitHub الحالي ({asset.get('size')}) لا يطابق الحجم المثبت ({SOURCE_BYTES}).")

ASSET_API_URL = f"https://api.github.com/repos/ZzwahwahzZ/Conquer-Mobile-Assets./releases/assets/{asset['id']}"
print("حجم المصدر المعلن:", asset["size"], "| أصل الإصدار:", asset["name"])
print("تجديد رابط التنزيل المؤقت والتحقق من Range...")
range_headers = dict(API_HEADERS)
range_headers["Accept"] = "application/octet-stream"
probe = session.get(ASSET_API_URL, headers={**range_headers, "Range":"bytes=0-0"},
                    allow_redirects=True, stream=True, timeout=60)
if probe.status_code != 206:
    status, detail = probe.status_code, probe.headers.get("Content-Range", "لا يوجد Content-Range")
    probe.close()
    raise RuntimeError(f"GitHub API لم يؤكد Range (HTTP {status}; {detail}). أوقفت التنفيذ دون تنزيل الأرشيف كاملًا.")
content_range = probe.headers.get("Content-Range", "")
probe.close()
if not content_range.startswith("bytes 0-0/"):
    raise RuntimeError("استجابة Range غير مطابقة؛ أوقفت التنفيذ لحماية وقتك وحجم البيانات.")

print("قراءة فهرس ZIP فقط، دون فك بقية الأرشيف...")
apk_reports, inventory_rows, text_rows, dex_candidates = [], [], [], []
with RemoteZip(ASSET_API_URL, session=session) as rz:
    names = set(rz.namelist())
    missing = sorted(TARGET_APKS - names)
    if missing:
        raise RuntimeError("ملفات APK المستهدفة غير موجودة: " + ", ".join(missing))
    selected = sorted(TARGET_APKS)
    print("ملفات APK المحددة:", len(selected))
    for index, apk_path in enumerate(selected, 1):
        print(f"[{index}/{len(selected)}] سحب وتحليل {apk_path}")
        apk_bytes = rz.read(apk_path)  # قراءة عضو APK فقط من الأرشيف البعيد
        apk_sha = sha256(apk_bytes)
        apk_name = Path(apk_path).name
        apk_info = {
            "archive_path": apk_path, "file": apk_name, "bytes": len(apk_bytes),
            "sha256": apk_sha, "inner_file_count": 0, "dex_count": 0,
            "text_files_read": 0, "keyword_candidate_count": 0,
            "categories": {}
        }
        with zipfile.ZipFile(io.BytesIO(apk_bytes)) as apk:
            infos = [i for i in apk.infolist() if not i.is_dir()]
            apk_info["inner_file_count"] = len(infos)
            for info in infos:
                name = info.filename
                category = classify(name)
                ext = Path(name).suffix.lower()
                inventory_rows.append({
                    "apk": apk_name, "path": name, "bytes": info.file_size,
                    "compressed_bytes": info.compress_size, "extension": ext,
                    "category": category, "crc32": f"{info.CRC:08x}"
                })
                apk_info["categories"][category] = apk_info["categories"].get(category, 0) + 1

                # نصوص DEX: استخراج سلاسل UTF-8/ASCII المطبوعة فقط، بلا فك أو تعديل للتطبيق.
                if re.fullmatch(r"classes\d*\.dex", Path(name).name):
                    dex = apk.read(info)
                    apk_info["dex_count"] += 1
                    strings = re.findall(rb"[\x20-\x7e]{5,}", dex)
                    for raw in strings:
                        s = raw.decode("utf-8", "ignore").strip()
                        hits = candidate_hits(s)
                        if hits:
                            dex_candidates.append({
                                "apk": apk_name, "dex": name,
                                "keywords": "|".join(hits), "candidate": s[:700]
                            })

                # قراءة الملفات النصية الصغيرة ذات الصلة فقط، مع حد أقصى واضح.
                if (ext in TEXT_EXT and info.file_size <= MAX_TEXT_FILE
                    and apk_info["text_files_read"] < MAX_TEXT_FILES_PER_APK
                    and (category != "other" or name.lower().startswith(("assets/","res/raw/","res/xml/")))):
                    try:
                        raw = apk.read(info)
                        decoded = raw.decode("utf-8-sig")
                        if decoded and "\x00" not in decoded[:2000]:
                            hits = candidate_hits(name + "\n" + decoded[:200000])
                            text_rows.append({
                                "apk": apk_name, "path": name,
                                "bytes": info.file_size, "category": category,
                                "keywords": "|".join(hits),
                                "content_preview": decoded[:12000]
                            })
                            apk_info["text_files_read"] += 1
                    except Exception:
                        pass
        apk_reports.append(apk_info)
        del apk_bytes

# ملفات نتائج مستقلة؛ لا نلمس تقارير Manifest السابقة.
generated = datetime.now(timezone.utc).isoformat()
summary = {
    "generated_utc": generated,
    "task": "focused_original_mobile_client_data_extraction",
    "scope": "Only the four known ConquerOnline APK members; no full archive extraction or previous manifest re-audit.",
    "source": {"release_tag": "mobile-extracted-assets", "asset_name": "extracted-assets-001.zip",
               "asset_id": asset["id"], "expected_bytes": SOURCE_BYTES, "expected_sha256": SOURCE_SHA256,
               "range_verified": True, "range_content_header": content_range, "full_archive_downloaded": False},
    "apk_files_analyzed": len(apk_reports),
    "apk_reports": apk_reports,
    "internal_file_inventory_count": len(inventory_rows),
    "text_files_read_count": len(text_rows),
    "dex_keyword_candidate_count": len(dex_candidates),
    "limitations": [
      "DEX strings are discovery clues, not proof of complete game-system behavior.",
      "Encrypted, compressed, proprietary or server-delivered data may require a separate format-specific stage.",
      "No source APK was modified."
    ]
}
(OUT/"client-data-summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
with (OUT/"apk-internal-inventory.csv").open("w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=["apk","path","bytes","compressed_bytes","extension","category","crc32"])
    w.writeheader(); w.writerows(inventory_rows)
with (OUT/"dex-system-keyword-candidates.csv").open("w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=["apk","dex","keywords","candidate"])
    w.writeheader(); w.writerows(dex_candidates)
with (OUT/"embedded-text-candidates.jsonl").open("w",encoding="utf-8") as f:
    for row in text_rows: f.write(json.dumps(row,ensure_ascii=False)+"\n")

# حزمة النتائج الصغيرة للتنزيل؛ لا تضم APKs ولا تعيد رفع ملفات العميل الأصلية.
bundle = Path("/content/Conquer-Mobile-Client-Data-Audit.zip")
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,p.name)

print("\nاكتمل التحليل الانتقائي.")
print("APK:",len(apk_reports),"| ملفات داخلية مفهرسة:",len(inventory_rows),
      "| نصوص مقروءة:",len(text_rows),"| مرشحات DEX:",len(dex_candidates))
print("حزمة النتائج:",bundle)

# رفع التقارير النصية الصغيرة إلى مجلد جديد في مستودع اللعبة (يتطلب GitHub PAT بصلاحية Contents: write).
# لا يتم رفع أي APK أو أصل أصلي أو ملف كبير.
token = getpass.getpass("GitHub PAT (Contents: Read and write) لرفع التقارير إلى مستودع اللعبة، أو Enter للتخطي: ").strip()
if token:
    api = f"https://api.github.com/repos/{TARGET_REPO}/contents"
    dest = "research/original-client-data-extraction"
    headers = {"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
    uploaded = []
    for p in sorted(OUT.iterdir()):
        if not p.is_file(): continue
        raw = p.read_bytes()
        if len(raw) > 8_000_000:
            print("تخطي ملف كبير:",p.name); continue
        path = f"{dest}/{p.name}"
        old = requests.get(f"{api}/{path}",headers=headers,params={"ref":"main"},timeout=45)
        payload = {"message":f"Add focused original client data audit: {p.name}",
                   "content":__import__("base64").b64encode(raw).decode("ascii"),
                   "branch":"main"}
        if old.status_code == 200:
            payload["sha"] = old.json()["sha"]
        elif old.status_code not in (404,):
            raise RuntimeError(f"تعذر قراءة حالة {path}: {old.status_code} {old.text[:300]}")
        put = requests.put(f"{api}/{path}",headers=headers,json=payload,timeout=90)
        if put.status_code not in (200,201):
            raise RuntimeError(f"فشل رفع {path}: {put.status_code} {put.text[:500]}")
        uploaded.append(path)
    print("تم رفع التقارير:",uploaded)
else:
    print("تم تخطي الرفع. التقارير موجودة في:",OUT)

try:
    from google.colab import files
    files.download(str(bundle))
except Exception:
    print("نزّل الملف يدويًا من:",bundle)
